# 卒論：改善版特徴量の再解析・段位別比較・固定モデル評価

**研究対象は「しょぼんｎ」本人の低段位期・高段位期の打ち方の違いです。**
旧23特徴量・H1〜H6・旧成果物は保持し、新しい出力先へ保存します。

- 段位16以下／18以上の定義を維持。段位17と初期1500観測対局は履歴に含め、採点から除外します。
- 主方式は `hour_invariant`。`order_verified=False`、`full_history_confirmed=False` を維持します。
- **このNotebookの配布時点では、全対局の改善版統計・実データ学習・外部性能は未実行です。**
- 記述分析を主目的とし、固定LightGBMは補助評価です。ウルトラ立直の結果に基づく選択・調整をしません。

**C1→C2→C3→C4を実行して件数・試験抽出を確認し、その後にC5以降を実行してください。**
初回から「すべてのセルを実行」は使わず、段階ごとに確認します。
C5とC8は不足分の牌譜解析、C7は研究用モデル学習、C9は外部採点を行います。
C10はC6直後にも実行でき、未実行の評価を数値で埋めません。

## C1：固定コード・依存関係・Drive接続・実装テスト

初回にColabの読み込み済みライブラリと固定版が異なる場合、導入後にPythonセッションを一度再起動します。
**接続が戻ったらC1をもう一度実行してください。** その後、Drive接続と実装テストへ進みます。
再起動によりセッション内の変数が消えるため、C1は分析セルより先に実行してください。Drive上の保存データは保持されます。
テストには合成データを使用します。


In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, json, os, re, subprocess, sys, urllib.request

CODE_COMMIT = "15572a5a3f21da470c4fcbb0e16496f31cba0b3e"  # 配布時に40桁の固定コミットへ置換
if not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise RuntimeError("固定コードのコミット未設定です。公開された完成版Notebookを開いてください。")
CODE_ROOT = Path("/content/mahjong_research_v2") / CODE_COMMIT
CODE_ROOT.mkdir(parents=True, exist_ok=True)
for name, module in tuple(sys.modules.items()):
    if name == "mahjong_research" or name.startswith("mahjong_research."):
        loaded_path = getattr(module, "__file__", None)
        if loaded_path and not Path(loaded_path).resolve().is_relative_to(CODE_ROOT.resolve()):
            raise RuntimeError("別コミットの研究コードが読込済みです。Colabセッションを再起動してC1を再実行してください。")
SOURCE_URL = f"https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{CODE_COMMIT}/research_v2/"
FILES = [
    "requirements.txt", "README.md", "legacy_feature_audit.csv",
    "feature_definitions_v2.csv", "old_to_v2_mapping.csv",
    "mahjong_research/__init__.py", "mahjong_research/legacy.py",
    "mahjong_research/tenhou.py", "mahjong_research/inputs.py",
    "mahjong_research/analysis.py", "mahjong_research/reporting.py",
    "mahjong_research/workflow.py", "tests/test_tenhou.py",
    "mahjong_research/recovery.py", "tests/test_disclosure_regression.py",
    "tests/fixtures/disclosed_own_four_tiles.xml",
    "tests/test_inputs.py", "tests/test_analysis.py", "tests/test_workflow.py",
    "docs/feature_audit_ja.md", "docs/thesis_draft_ja.md",
    "docs/extractor_validation.json", "docs/extractor_validation.md",
    "docs/analysis_validation.json", "docs/analysis_validation_ja.md",
]
manifest_path = CODE_ROOT / "bootstrap_manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
downloaded = 0
for relative in FILES:
    dest = CODE_ROOT / relative
    if dest.is_file() and manifest.get(relative) == hashlib.sha256(dest.read_bytes()).hexdigest():
        continue
    with urllib.request.urlopen(SOURCE_URL + relative, timeout=60) as response:
        payload = response.read()
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(payload)
    manifest[relative] = hashlib.sha256(payload).hexdigest()
    downloaded += 1
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

expected = {}
for line in (CODE_ROOT / "requirements.txt").read_text().splitlines():
    if "==" in line and not line.strip().startswith("#"):
        package, version = line.strip().split("==", 1)
        expected[package] = version
module_names = {"scikit-learn": "sklearn"}
loaded_mismatch = []
need_install = False
for package, version in expected.items():
    module = sys.modules.get(module_names.get(package, package))
    if module is not None and getattr(module, "__version__", None) != version:
        loaded_mismatch.append(package)
    try:
        need_install |= importlib.metadata.version(package) != version
    except importlib.metadata.PackageNotFoundError:
        need_install = True
if need_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements.txt")])
if loaded_mismatch:
    # Native libraries already imported by Colab need a fresh Python process.
    # A persistent guard prevents an automatic restart loop.
    restart_guard = CODE_ROOT / "dependency_restart.json"
    previous = json.loads(restart_guard.read_text(encoding="utf-8")) if restart_guard.exists() else None
    if previous and previous.get("expected") == expected and not need_install:
        raise SystemExit("再起動後もライブラリの版が一致しません。［ランタイム］からセッションを再起動してC1を再実行してください。改善しない場合はC1の出力を共有してください。対象: " + ", ".join(loaded_mismatch))
    restart_guard.write_text(json.dumps({"expected": expected, "pid": os.getpid()}, indent=2), encoding="utf-8")
    print("依存関係の準備が完了しました。読み込み済みライブラリを切り替えるため、Pythonセッションを一度再起動します。", flush=True)
    print("接続が戻ったらC1をもう一度実行してください。次にテストとDrive接続へ進みます。", flush=True)
    os.kill(os.getpid(), 9)  # SIGKILL on the Linux Colab runtime
    raise SystemExit("セッションの再起動を待ち、C1を再実行してください。")
for package, version in expected.items():
    if importlib.metadata.version(package) != version:
        raise RuntimeError(f"依存関係が一致しません: {package}=={version}")

os.environ["MPLCONFIGDIR"] = str(CODE_ROOT / ".matplotlib")
sys.path.insert(0, str(CODE_ROOT)) if str(CODE_ROOT) not in sys.path else None
from google.colab import drive
drive.mount("/content/drive")
import pandas as pd
from IPython.display import display, Markdown
from mahjong_research.analysis import AnalysisBlocked, load_frozen_bundle
from mahjong_research.workflow import (
    initialize, preview, pilot, extract_stage, descriptive_stage,
    model_stage, external_stage, final_report,
)
import unittest
suite = unittest.defaultTestLoader.discover(str(CODE_ROOT / "tests"), pattern="test_*.py")
test_result = unittest.TextTestRunner(verbosity=1).run(suite)
if not test_result.wasSuccessful():
    raise RuntimeError("実装テストが失敗しました。実データ解析へ進まないでください。")
print("固定コード:", CODE_COMMIT, "今回の取得:", downloaded, "ファイル")
print("実装テスト完了。実データの全件抽出・モデル学習は未実行です。")


## C2：入力パス・新しい実行フォルダ

ZIP名が異なる場合だけ実際のパスに変更します。旧キャッシュ候補が複数ある場合は
`LEGACY_DIRS` に `raw_manifest.csv` と `raw_features.csv` のあるフォルダを指定します。
再開時は前回表示された `RUN_DIR` を `RESUME_RUN_DIR` に指定してください。
保存済み旧CSVを新版として読み替えることはしません。

In [ ]:
ROOT = Path("/content/drive/MyDrive/卒論")
ZIP_PATHS = {
    "shobon": ROOT / "mjlog_pf4-20_n30.zip",
    "ultra": ROOT / "ultrareach.zip",
}
LEGACY_DIRS = {
    "shobon": None,  # 例: ROOT / "validation_inputs/zip_cache/shobon_旧署名"
    "ultra": None,
}
RESUME_RUN_DIR = None  # 再開時だけ前回RUN_DIRのパス文字列を指定
ctx = initialize(ROOT, CODE_ROOT, CODE_COMMIT, ZIP_PATHS, LEGACY_DIRS, resume=RESUME_RUN_DIR)
RUN_DIR = Path(ctx["folder"])
print("RUN_DIR =", RUN_DIR)
print("order_verified=False / full_history_confirmed=False")

# Reset in-memory results so another run cannot leak into this report.
old_desc = new_desc = legacy_bundle = new_bundle = external_metrics = None
old = plans = shared = new_shobon = new_ultra = None


## C3：入力監査・共通対局・処理件数の事前表示

旧CSV・キャッシュの署名とZIP一覧を照合します。牌譜の全件再解析は行いません。
不足ファイルや未解決要素が表示された場合は、その内容を確認してください。
共通対局を含む窓は学習から除外し、履歴の行を詰め直しません。

In [ ]:
old, plans, shared = preview(ctx)
display(pd.DataFrame([
    {"player": key, "旧特徴量対局数": len(old[key]),
     "抽出対象数": len(plan["members"]), "新版キャッシュ済み": plan["already_cached"],
     "未抽出数": plan["remaining"], "旧アーカイブ未解決要素": plan["legacy_unresolved_entries"],
     "新版キャッシュ保存先": plan["cache_dir"]}
    for key, plan in plans.items()
]))
print("両プレイヤーに共通する対局ID:", len(shared), "件")


## C3R：今回の1対局エラーから復旧する場合だけ実行
C1・C2・C3の後で実行します。C2は新しい実行フォルダを作成してください。旧抽出器と全バッチ・入力の署名が一致する場合だけ正常行を新しいキャッシュへ移行します。旧キャッシュは保持されます。表示が「正常11572、残り1」ならC5へ進んでください。通常の初回解析ではこのセルを飛ばします。


In [ ]:
from mahjong_research.recovery import recover_successful_batches
OLD_SHOBON_CACHE = ROOT / "research_v2_cache/shobon_4105129b8f8dbace9d04"
plans["shobon"] = recover_successful_batches(plans["shobon"], OLD_SHOBON_CACHE)
print("署名照合済みの正常キャッシュ:", plans["shobon"]["already_cached"])
print("再解析する対象数:", plans["shobon"]["remaining"])
print("新しい保存先:", plans["shobon"]["cache_dir"])


## C4：しょぼんｎの少数牌譜で試験抽出

未抽出から最大8件を処理します。既存の新版キャッシュは再利用します。
手牌・イベント整合性の失敗があれば、その対局を黙って削除せず停止します。
この試験が成功しても、全件の正しさを確認したことにはなりません。

In [ ]:
pilot_audit = pilot(ctx, plans["shobon"], max_new=8)
display(pilot_audit)
print("少数確認の範囲です。全件の品質確認はC5で行います。")


## C5：しょぼんｎの新版特徴量を全件生成・検査

**このセルで未抽出分の牌譜解析を実行します。** C3/C4の結果を確認してから進んでください。
途中で切断されても、同じコード・入力の署名付きバッチキャッシュを再利用できます。
初期期間・段位17を含む全対局を保持し、補完前・移動平均前の新版特徴量と分子分母を保存します。

In [ ]:
new_shobon, quality_shobon = extract_stage(ctx, plans["shobon"])
display(quality_shobon)
print("しょぼんｎ 新版対局数:", len(new_shobon))


## C6：旧版・新版の段位別比較、分布、月別推移

同じ終端・元対局集合の `hour_invariant` 窓で比較します。
**記述分析は補完せず、特徴量ごとに10対局すべてに値がある窓だけ平均を定義します。**
希少行動では有効窓が不足する場合があるため、欠損率、対局単位、非重複窓、機会数も確認します。
ここでの分布はH4/H5の補完済み入力分布と一致するとは限りません。
重複窓を独立標本とみなした検定は行いません。

In [ ]:
old_desc, new_desc, descriptive_out = descriptive_stage(
    ctx, old["shobon"], new_shobon, shared, make_plots=True
)
display(pd.DataFrame([{"version": "legacy23", **old_desc["summary"]},
                      {"version": "v2", **new_desc["summary"]}]))
display(new_desc["differences"].query("view == 'primary_hour_invariant'")
        .sort_values("abs_smd", ascending=False))
print("比較図表・日本語レポート:", descriptive_out)
print("記述分析のみで終える場合も、C10を実行して実行状態を記録できます。")


## C6B：分子・分母を合算する追加検証
C6後に実行します。このセルだけを現在実行中のNotebookへコピーして使えます。新しいセッションではC1後にC6_REFERENCE_DIRへ前回のC6保存先を指定すれば、C2〜C6を再実行せず解析できます。キャッシュから読み取り、追加結果は卒論/research_v2_opportunity/<新実行ID>へ保存します。
18割合と2イベント加重平均を、従来方式・同じ元対局非重複窓・共通有効窓で比較します。改善項目数と差の安定性はこのセルの実データ出力で確認してください。C7へ進む前にreport_ja.mdと欠損・機会数を確認してください。


In [ ]:
# C6B: run after C6, or set the saved C6 directory below after C1.
from pathlib import Path
import hashlib, importlib.util, json, sys, urllib.request
import pandas as pd
from IPython.display import display, Markdown

SUPPLEMENT_COMMIT = "6c6b9f46ae39c615c38409e3d9e36f37297a9909"
C6_REFERENCE_DIR = None  # e.g. "/content/drive/MyDrive/卒論/research_v2_runs/.../descriptive/..."
if C6_REFERENCE_DIR is None:
    C6_REFERENCE_DIR = globals().get("descriptive_out")
if C6_REFERENCE_DIR is None:
    raise ValueError("C6_REFERENCE_DIRに前回C6が表示した保存先（descriptive/<実行ID>）を設定してください。")
reference = Path(C6_REFERENCE_DIR)
run_dir = reference.parent.parent
reference_meta_path = reference / "v2" / "primary_windows.csv"
for required in [reference_meta_path, run_dir/"input_plan.json", run_dir/"shared_games.csv"]:
    if not required.is_file():
        raise FileNotFoundError(f"C6参照ファイルがありません: {required}")
run_config = json.loads((run_dir/"run_configuration.json").read_text(encoding="utf-8"))
if run_config["code_commit"] != CODE_COMMIT:
    raise ValueError("C6の固定コードとC1のCODE_COMMITが違います。元のC6と同じコードを使用してください。")
cache_dir = Path(json.loads((run_dir/"input_plan.json").read_text(encoding="utf-8"))["shobon"]["cache_dir"])
raw_path = cache_dir / "raw_features_v2.csv"
complete = json.loads((cache_dir/"complete.json").read_text(encoding="utf-8"))
from mahjong_research import tenhou as extractor
if (hashlib.sha256(raw_path.read_bytes()).hexdigest() != complete["sha256"]
        or hashlib.sha256(Path(extractor.__file__).read_bytes()).hexdigest() != complete["extractor_sha256"]
        or complete.get("order_verified") is not False or complete.get("full_history_confirmed") is not False):
    raise ValueError("C6キャッシュのハッシュ・抽出器・確認フラグの照合に失敗しました。")
data_for_supplement = pd.read_csv(raw_path, float_precision="round_trip")
if len(data_for_supplement) != complete["rows"]:
    raise ValueError("C6キャッシュの対局件数が一致しません。")
reference_meta = pd.read_csv(reference_meta_path)
shared_for_supplement = set(pd.read_csv(run_dir/"shared_games.csv").game_id)
supplement_root = Path("/content/mahjong_opportunity_supplement") / SUPPLEMENT_COMMIT
supplement_root.mkdir(parents=True, exist_ok=True)
for filename in ["opportunity_analysis.py", "test_opportunity_analysis.py"]:
    url=f"https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{SUPPLEMENT_COMMIT}/research_v2/supplement/{filename}"
    payload=urllib.request.urlopen(url, timeout=60).read()
    (supplement_root/filename).write_bytes(payload)
module_name = "opportunity_analysis"
spec=importlib.util.spec_from_file_location(module_name, supplement_root/"opportunity_analysis.py")
module=importlib.util.module_from_spec(spec)
sys.modules[module_name]=module
spec.loader.exec_module(module)
import unittest
test_spec=importlib.util.spec_from_file_location("test_opportunity_analysis",supplement_root/"test_opportunity_analysis.py")
test_module=importlib.util.module_from_spec(test_spec)
test_spec.loader.exec_module(test_module)
test_result=unittest.TextTestRunner(verbosity=1).run(unittest.defaultTestLoader.loadTestsFromModule(test_module))
if not test_result.wasSuccessful():
    raise RuntimeError("追加集計のテストに失敗しました。入力を変更せず停止します。")
drive_root=Path(run_config["root"])
opportunity_result=module.run_opportunity_analysis(data_for_supplement,shared_for_supplement,reference_meta,
                                                  drive_root/"research_v2_opportunity")
display(pd.DataFrame([opportunity_result["facts"]]))
display(opportunity_result["comparisons"])
display(Markdown((opportunity_result["output_dir"]/"report_ja.md").read_text(encoding="utf-8")))
print("追加分析の保存先:", opportunity_result["output_dir"])


## C7：固定LightGBMの内部評価と外部評価前のモデル固定

**このセルで研究用モデルを学習します。** 固定200木・深さ5・学習率0.03等を使用し、探索や早期終了を行いません。
訓練区間だけで補完値を推定し、窓の元対局重複と共有対局混入を検査します。
学習が単一クラスの分割は停止・記録し、単一クラス評価のAUC／Balanced Accuracyは算出しません。
旧版の学習前提が満たされない場合は理由を保存し、新版の処理を別に試します。
同じ実行フォルダに固定モデルがあれば、検証して再利用します。

In [ ]:
legacy_bundle = None
new_bundle = None
try:
    legacy_dir = RUN_DIR / "model_legacy"
    if legacy_dir.exists() and any(legacy_dir.iterdir()) and not (legacy_dir / "external_freeze.json").exists():
        raise AnalysisBlocked("Previous legacy model attempt is incomplete; preserve its audit and continue only v2.")
    legacy_bundle = model_stage(ctx, old["shobon"], shared, version="legacy")
    display(legacy_bundle["fold_audit"])
    display(legacy_bundle["metrics"])
except AnalysisBlocked as exc:
    from datetime import datetime, timezone
    import uuid
    status_dir = RUN_DIR / "status"
    status_dir.mkdir(exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    status_path = status_dir / f"legacy_model_blocked_{stamp}.json"
    status_path.write_text(json.dumps({"status": "blocked", "reason": str(exc),
        "note": "旧版性能は未算出。数値を補わない。"}, ensure_ascii=False, indent=2), encoding="utf-8")
    print("旧版の学習を停止:", exc, "\n記録:", status_path)
new_bundle = model_stage(ctx, new_shobon, shared, version="v2")
display(new_bundle["fold_audit"])
display(new_bundle["metrics"])
print("外部評価前の固定モデル:", new_bundle["model_dir"])


## C8：ウルトラ立直の新版特徴量生成・検査

**このセルで外部プレイヤーの未抽出分を解析します。**
C7で固定した新版モデルの記録・前処理を検証してから処理を開始します。
外部プレイヤーの数値に基づく特徴量選択、パラメータ・閾値変更は行いません。

In [ ]:
if globals().get("new_bundle") is None:
    raise RuntimeError("C7で新版モデルを固定・再読込してからC8を実行してください。")
new_bundle = load_frozen_bundle(new_bundle["model_dir"])
new_ultra, quality_ultra = extract_stage(ctx, plans["ultra"])
display(quality_ultra)
print("ウルトラ立直 新版対局数:", len(new_ultra))


## C9：固定モデルの外部評価

固定モデル・訓練補完値で一度採点します。外部データでは学習しません。
完了済みの同一入力・モデル・出力先なら、保存済み結果だけを読み込みます。
途中で失敗した外部評価は黙ってやり直さず、記録を確認します。

In [ ]:
external_metrics = external_stage(ctx, new_bundle, new_ultra)
display(external_metrics)
print("外部評価のクラス構成・混同行列・予測スコア分布も保存しました。")


## C10更新版：機会数方式を主分析候補とする卒論レポート
C6B・C7・C9の保存済みCSVを読み取り、新しいfinal_report_opportunity/<実行ID>へ保存します。主分析候補は機会数方式、従来単純平均は感度分析、C7/C9は従来入力による補助分析として掲載します。窓内合算後に窓を等重みで比較する方法を明記します。
現在の実行中NotebookへこのC10セルをコピーして実行できます。新しいセッションではC1後、セル冒頭の2つのパスを指定してC10だけ実行してください。C2〜C9の再実行は不要です。
序盤ノーテン非現物率を含め、数値はC6Bの実際のCSVから取得します。旧C10出力を保持します。


In [ ]:
# Updated C10: read saved C6B/C7/C9 results only.
from pathlib import Path
import importlib.util,sys,urllib.request,unittest
from IPython.display import display,Markdown

REPORT_COMMIT="5d1a50ff40e2ab6c85a7bb3385caa14495900e11"
OPPORTUNITY_DIR=None  # C6B output, e.g. /content/drive/MyDrive/卒論/research_v2_opportunity/<ID>
RUN_DIR_FOR_REPORT=None  # C7/C9 run folder, e.g. .../research_v2_runs/<ID>
if OPPORTUNITY_DIR is None:
    OPPORTUNITY_DIR=(globals().get("opportunity_result") or {}).get("output_dir")
if RUN_DIR_FOR_REPORT is None:
    RUN_DIR_FOR_REPORT=(globals().get("ctx") or {}).get("folder") or globals().get("RUN_DIR")
if OPPORTUNITY_DIR is None or RUN_DIR_FOR_REPORT is None:
    raise ValueError("OPPORTUNITY_DIRにC6B保存先、RUN_DIR_FOR_REPORTにC7/C9の実行フォルダを設定してください。")
report_code_root=Path("/content/mahjong_opportunity_report")/REPORT_COMMIT
report_code_root.mkdir(parents=True,exist_ok=True)
for name in ["opportunity_report.py","test_opportunity_report.py"]:
    url=f"https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{REPORT_COMMIT}/research_v2/supplement/{name}"
    (report_code_root/name).write_bytes(urllib.request.urlopen(url,timeout=60).read())
spec=importlib.util.spec_from_file_location("opportunity_report",report_code_root/"opportunity_report.py")
report_module=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=report_module
spec.loader.exec_module(report_module)
spec_test=importlib.util.spec_from_file_location("test_opportunity_report",report_code_root/"test_opportunity_report.py")
report_tests=importlib.util.module_from_spec(spec_test)
spec_test.loader.exec_module(report_tests)
checks=unittest.TextTestRunner(verbosity=1).run(unittest.defaultTestLoader.loadTestsFromModule(report_tests))
if not checks.wasSuccessful():raise RuntimeError("更新レポートの回帰テスト失敗。保存済み入力は変更せず停止します。")
updated_c10_result=report_module.updated_thesis_report(OPPORTUNITY_DIR,RUN_DIR_FOR_REPORT)
display(updated_c10_result["comparison"])
display(updated_c10_result["performance"])
display(Markdown(updated_c10_result["report_path"].read_text(encoding="utf-8")))
print("更新版C10の保存先:",updated_c10_result["output_dir"])
